# V2 Identifiability & Evaluation Analysis
Analyzing results from run: `20261002_151931_judge_llama-3.3-70b-instruct-fp8_base_gemma-3-27b-it`
Focusing on the updated Identifiability (v2) metric.

In [16]:
import pandas as pd
import numpy as np
import os
import json

RUN_DIR = '/home/gpuuser7/gpuuser7_a/prateek/GEM-Bench/GemBench/benchmarking/output/evaluate_results/output/20261002_151931_judge_llama-3.3-70b-instruct-fp8_base_gemma-3-27b-it'
IDENT_FILE = os.path.join(RUN_DIR, 'identifiability_evaluation.xlsx')
AVG_FILE = os.path.join(RUN_DIR, 'evaluation_result_average.xlsx')

## 1. Identifiability Analysis (V2 Prompt)

In [17]:
df_ident = pd.read_excel(IDENT_FILE, skiprows=1, header=None)
df_ident.columns = ['question', 'method', 'response', 'analysis', 'category', 'score']

print("Mean Identifiability Score by Method (Higher = Better/More identifiable):")
display(df_ident.groupby('method')['score'].mean().round(2).sort_values(ascending=False).to_frame())

print("\nCategory Distribution by Method (%):")
ct = pd.crosstab(df_ident['method'], df_ident['category'], normalize='index') * 100
order = ['NOT_IDENTIFIABLE', 'SOMEWHAT_IDENTIFIABLE', 'IDENTIFIABLE', 'HIGHLY_IDENTIFIABLE']
display(ct[[c for c in order if c in ct.columns]].round(1))

Mean Identifiability Score by Method (Higher = Better/More identifiable):


,score
method,
GI-R,44.37
GI-P,44.21
GIR-R,43.06
GIR-P,40.36
Ad-Chat,35.56



Category Distribution by Method (%):


category,NOT_IDENTIFIABLE,SOMEWHAT_IDENTIFIABLE,IDENTIFIABLE,HIGHLY_IDENTIFIABLE
method,,,,
Ad-Chat,5.8,83.6,7.7,2.9
GI-P,2.3,68.0,23.3,6.4
GI-R,2.9,68.6,20.0,8.6
GIR-P,3.0,75.8,17.1,4.1
GIR-R,2.3,71.6,19.6,6.5


In [18]:
# Compare GI-R vs GIR-R specifically
pivot_score = df_ident.pivot_table(index='question', columns='method', values='score')
pivot_cat = df_ident.pivot_table(index='question', columns='method', values='category', aggfunc='first')

if 'GI-R' in pivot_score.columns and 'GIR-R' in pivot_score.columns:
    gi_r = pivot_score['GI-R']
    gir_r = pivot_score['GIR-R']
    
    print(f"Questions where GI-R scored HIGHER (better): {(gi_r > gir_r).sum()}")
    print(f"Questions where GIR-R scored HIGHER: {(gi_r < gir_r).sum()}")
    print(f"Questions with SAME score: {(gi_r == gir_r).sum()}")
    
    diff = pivot_cat[pivot_cat['GI-R'] != pivot_cat['GIR-R']]
    print(f"\nQuestions where they fell into DIFFERENT categories: {len(diff)} / {len(pivot_cat)}")


Questions where GI-R scored HIGHER (better): 68
Questions where GIR-R scored HIGHER: 52
Questions with SAME score: 103

Questions where they fell into DIFFERENT categories: 76 / 223


## 2. Overall Performance Analysis (General Metrics)

In [26]:
# The format changed slightly, so we read the average metrics directly from the xlsx
df_avg = pd.read_excel(AVG_FILE, skiprows=1)
headers = list(df_avg.iloc[0])
df_avg.columns = [str(h) if pd.notna(h) else str(c) for c, h in zip(df_avg.columns, headers)]
df_avg = df_avg.iloc[1:].copy()
df_avg['data_set'] = df_avg['data_set'].ffill()

# Filter to the main methods of interest
methods_to_compare = ['GIR-P', 'Ad-Chat', 'GI-R', 'GIR-R', 'GI-P']
df_avg = df_avg[df_avg['solution'].isin(methods_to_compare)].copy()

qual_cols = ['identifiability']
qual_cols = [c for c in qual_cols if c in df_avg.columns]

# print("Qualitative Metrics (GI-R vs GIR-R):")
df_qual = df_avg[['data_set', 'solution'] + qual_cols].copy()
for col in qual_cols:
    df_qual[col] = pd.to_numeric(df_qual[col], errors='coerce').round(2)
display(df_qual.set_index(['data_set', 'solution']))


identifiability
data_set  solution                 
CA_Prod   Ad-Chat             32.93
          GI-R                36.16
          GI-P                36.46
          GIR-R               33.54
          GIR-P               32.73
LM-Market Ad-Chat             37.58
          GI-R                51.66
          GI-P                51.39
          GIR-R               52.40
          GIR-P               48.27
MT-Human  Ad-Chat             47.50
          GI-R                68.60
          GI-P                64.10
          GIR-R               60.77
          GIR-P               52.97

## 3. Latest Run Analysis: Qwen3-32B
Analyzing results from the latest run with Qwen3-32B as the base model:
`20261004_174237_judge_llama-3.3-70b-instruct-fp8_base_qwen3-32b`

In [20]:
RUN_DIR_LATEST = '/home/gpuuser7/gpuuser7_a/prateek/GEM-Bench/GemBench/benchmarking/output/evaluate_results/output/20261004_174237_judge_llama-3.3-70b-instruct-fp8_base_qwen3-32b'
IDENT_FILE_LATEST = os.path.join(RUN_DIR_LATEST, 'identifiability_evaluation.xlsx')

if os.path.exists(IDENT_FILE_LATEST):
    df_ident_latest = pd.read_excel(IDENT_FILE_LATEST, skiprows=1, header=None)
    df_ident_latest.columns = ['question', 'method', 'response', 'analysis', 'category', 'score']

    print("=== Mean Identifiability Score by Method (Qwen3-32B) ===")
    display(df_ident_latest.groupby('method')['score'].mean().round(2).sort_values(ascending=False).to_frame())
else:
    print(f"File not found: {IDENT_FILE_LATEST}")

=== Mean Identifiability Score by Method (Qwen3-32B) ===


,score
method,
GI-R,47.27
GIR-P,41.32
GIR-R,40.41
Ad-Chat,37.49


In [21]:
if os.path.exists(IDENT_FILE_LATEST):
    print("=== Category Distribution by Method (%) (Qwen3-32B) ===")
    ct_latest = pd.crosstab(df_ident_latest['method'], df_ident_latest['category'], normalize='index') * 100
    order = ['NOT_IDENTIFIABLE', 'SOMEWHAT_IDENTIFIABLE', 'IDENTIFIABLE', 'HIGHLY_IDENTIFIABLE']
    display(ct_latest[[c for c in order if c in ct_latest.columns]].round(1))


=== Category Distribution by Method (%) (Qwen3-32B) ===


category,NOT_IDENTIFIABLE,SOMEWHAT_IDENTIFIABLE,IDENTIFIABLE,HIGHLY_IDENTIFIABLE
method,,,,
Ad-Chat,19.7,61.2,5.4,13.8
GI-R,6.4,55.2,27.5,10.9
GIR-P,1.9,71.3,26.5,0.3
GIR-R,1.9,73.9,24.1,0.1


In [22]:
if os.path.exists(IDENT_FILE_LATEST):
    print("=== GI-R vs GIR-R Score Delta (Qwen3-32B) ===")
    pivot_score_latest = df_ident_latest.pivot_table(index='question', columns='method', values='score')
    
    if 'GI-R' in pivot_score_latest.columns and 'GIR-R' in pivot_score_latest.columns:
        gi_r_latest = pivot_score_latest['GI-R']
        gir_r_latest = pivot_score_latest['GIR-R']
        
        print(f"Questions where GI-R scored HIGHER (better): {(gi_r_latest > gir_r_latest).sum()}")
        print(f"Questions where GIR-R scored HIGHER: {(gi_r_latest < gir_r_latest).sum()}")
        print(f"Questions with SAME score: {(gi_r_latest == gir_r_latest).sum()}")


=== GI-R vs GIR-R Score Delta (Qwen3-32B) ===
Questions where GI-R scored HIGHER (better): 108
Questions where GIR-R scored HIGHER: 63
Questions with SAME score: 52


In [23]:
import json, os, glob, pandas as pd
if os.path.exists(IDENT_FILE_LATEST):
    DATASET_DIR = '/home/gpuuser7/gpuuser7_a/prateek/GEM-Bench/GemBench/benchmarking/dataset'
    q_to_ds = {}
    for root, _, files in os.walk(DATASET_DIR):
        for f in files:
            if f.endswith('.json'):
                path = os.path.join(root, f)
                dataset_name = os.path.basename(os.path.dirname(path))
                with open(path, 'r') as jf:
                    data = json.load(jf)
                    for entry in data:
                        if isinstance(entry, str):
                            q_to_ds[entry] = dataset_name
                        elif isinstance(entry, dict) and 'question' in entry:
                            q_to_ds[entry['question']] = dataset_name
                        elif isinstance(entry, dict) and 'prompt' in entry:
                            q_to_ds[entry['prompt']] = dataset_name
    
    def map_q(q_excel):
        q_excel = str(q_excel).strip()
        for q_dataset, ds in q_to_ds.items():
            if q_excel[:40] in q_dataset or q_dataset[:40] in q_excel:
                return ds
        return 'CA_Prod'  # Catch-all for remaining CA_Prod questions
    
    df_ident_latest['dataset'] = df_ident_latest['question'].apply(map_q)
    
    print("=== Mean Identifiability Score by Dataset & Method (Qwen3-32B) ===")
    score_ds = df_ident_latest.pivot_table(index='dataset', columns='method', values='score', aggfunc='mean')
    display(score_ds.round(2))


=== Mean Identifiability Score by Dataset & Method (Qwen3-32B) ===


method,Ad-Chat,GI-R,GIR-P,GIR-R
dataset,,,,
CA_Prod,27.32,42.58,37.67,36.30
LM_Market,44.11,50.09,44.78,44.77
MT_Human,93.33,75.27,50.60,46.20
